In [8]:
import pandas as pd


# LOAD DATASET

df = pd.read_csv("Event_Booking_Details.csv")

# BASIC EDA

print(df.head())


print(df.tail())


print(df.shape)


print(df.columns)


print(df.dtypes)




print(df.isnull().sum())


print(df.duplicated().sum())

# DATA CLEANING


# Remove duplicate rows
df = df.drop_duplicates()

# Fill missing values in numeric columns
num_cols = df.select_dtypes(include=["number"]).columns

for col in num_cols:
    df[col] = df[col].fillna(0)

# Fill missing values in text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    df[col] = df[col].fillna("Unknown")

# DATE CONVERSION


df["Created Date"] = pd.to_datetime(
    df["Created Date"],
    errors="coerce"
)


# DERIVED FEATURES

df["Month"] = df["Created Date"].dt.month

df["Year"] = df["Created Date"].dt.year

df["Revenue Per Ticket"] = (
    df["Grand Total"] / df["Ticket Count"]
)

df["Net Revenue"] = (
    df["Grand Total"] - df["Refund Amount"]
)

# Handle division by zero
df["Revenue Per Ticket"] = (
    df["Revenue Per Ticket"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)


# FINAL CHECKS


print("\n MISSING VALUES AFTER CLEANING ")
print(df.isnull().sum())

print("\n FINAL SHAPE ")
print(df.shape)

print("\n FINAL DATA TYPES ")
print(df.dtypes)


# DATA DICTIONARY
print("DATA DICTIONARY")
dictionary_df = pd.DataFrame({
    "Column": [
        "Booking No",
        "Ticket Count",
        "Ticket Amount",
        "Grand Total",
        "Processing Fee",
        "Platform Fee",
        "Event Name",
        "Organizer",
        "Payment Status",
        "Booking Status",
        "Stripe Revenue",
        "Eventox Revenue",
        "Organizer Revenue",
        "Refund Amount",
        "Created Date",
        "Month",
        "Year",
        "Revenue Per Ticket",
        "Net Revenue"
    ],
    "Description": [
        "Unique booking ID",
        "Number of tickets booked",
        "Amount of ticket purchased",
        "Total amount paid",
        "Payment processing fee",
        "Eventox platform fee",
        "Name of the event",
        "Event organizer",
        "Payment status",
        "Booking status",
        "Revenue from Stripe",
        "Revenue earned by Eventox",
        "Revenue earned by organizer",
        "Amount refunded",
        "Booking creation date",
        "Month extracted from Created Date",
        "Year extracted from Created Date",
        "Grand Total divided by Ticket Count",
        "Grand Total minus Refund Amount"
    ]
})


print(dictionary_df)


# SAVE OUTPUT FILES


# Save cleaned dataset
df.to_csv("Event_Booking_Details_cleaned.csv", index=False)

# Save data dictionary
dictionary_df.to_csv("data_dictionary.csv", index=False)
print("-------------------------------------------------------------------")

print("DATA CLEANING COMPLETED SUCCESSFULLY")

print("Cleaned Dataset Saved As : Event_Booking_Details_cleaned.csv")
print("Data Dictionary Saved As : data_dictionary.csv")

              Booking No   Ticket Type  Ticket Count  Ticket Amount  \
0  EV1748739848135F6K005       Student             1           30.0   
1  EV1748758580099VGME5F        Adults             3          111.0   
2  EV1748763708930R7SJLB        Adults             1           37.0   
3  EV1748771174918ZLCXRL  Couple Entry             2           68.0   
4  EV17487734560106AU1PF        Adults             4          120.0   

   Processing Fee  Ticket Fee  Platform Fee  Agent Fee  Discount Amount  \
0            0.75         0.5             0          0              0.0   
1            2.78         1.5             0          0              0.0   
2            0.93         0.5             0          0              0.0   
3            1.70         1.0             0          0              0.0   
4            3.00         2.0             0          0              0.0   

   Grand Total  ... Booking Status  \
0        31.25  ...      confirmed   
1       115.28  ...      confirmed   
2       

In [ ]:
pip install scikit-learn

In [10]:
#use case study
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# =====================================
# LOAD DATASET
# =====================================

df = pd.read_csv("Event_Booking_Details_cleaned.csv")

# =====================================
# DATE CONVERSION
# =====================================

df["Created Date"] = pd.to_datetime(
    df["Created Date"],
    errors="coerce"
)

# Remove invalid dates
df = df.dropna(subset=["Created Date"])

# =====================================
# FEATURE ENGINEERING
# =====================================

# Month Number
df["Month"] = df["Created Date"].dt.month

# Day of Week
df["Weekday"] = df["Created Date"].dt.dayofweek

# Historical Bookings
df["Historical_Bookings"] = df["Ticket Count"]

# Past Sales Velocity (7-day moving average)
df["Sales_Velocity"] = (
    df["Ticket Count"]
    .rolling(window=7, min_periods=1)
    .mean()
)

# =====================================
# BOOKING FORECAST MODEL
# =====================================

features = [
    "Month",
    "Weekday",
    "Historical_Bookings",
    "Sales_Velocity"
]

X = df[features]
y = df["Ticket Count"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

booking_model = LinearRegression()

booking_model.fit(X_train, y_train)

booking_predictions = booking_model.predict(X_test)

print("\n========== BOOKING MODEL ==========")

print(
    "MAE:",
    round(
        mean_absolute_error(
            y_test,
            booking_predictions
        ),
        2
    )
)

print(
    "R2 Score:",
    round(
        r2_score(
            y_test,
            booking_predictions
        ),
        2
    )
)

# =====================================
# NEXT 7 DAYS BOOKING FORECAST
# =====================================

latest_booking = df["Ticket Count"].iloc[-1]
latest_velocity = df["Sales_Velocity"].iloc[-1]

future_rows = []

for i in range(1, 8):

    future_date = (
        df["Created Date"].max()
        + pd.Timedelta(days=i)
    )

    future_rows.append({
        "Month": future_date.month,
        "Weekday": future_date.dayofweek,
        "Historical_Bookings": latest_booking,
        "Sales_Velocity": latest_velocity
    })

future_df = pd.DataFrame(future_rows)

future_bookings = booking_model.predict(future_df)

print("\n========== NEXT 7 DAYS BOOKINGS ==========")

for i, value in enumerate(future_bookings, start=1):
    print(
        f"Day {i}:",
        round(value)
    )

expected_bookings = round(
    future_bookings.sum()
)

print(
    "\nExpected Bookings Next 7 Days:",
    expected_bookings
)

# =====================================
# REVENUE FORECAST
# =====================================

average_ticket_price = (
    df["Ticket Amount"].mean()
)

expected_revenue = (
    expected_bookings
    * average_ticket_price
)

print(
    "Expected Revenue: ₹",
    round(expected_revenue, 2)
)

# =====================================
# SAVE FORECAST RESULTS
# =====================================

forecast_df = pd.DataFrame({
    "Day": range(1, 8),
    "Predicted Bookings":
        np.round(future_bookings)
})

forecast_df.to_csv(
    "booking_forecast.csv",
    index=False
)

print(
    "\nForecast saved as booking_forecast.csv"
)


========== BOOKING MODEL ==========
MAE: 0.0
R2 Score: 1.0

========== NEXT 7 DAYS BOOKINGS ==========
Day 1: 2
Day 2: 2
Day 3: 2
Day 4: 2
Day 5: 2
Day 6: 2
Day 7: 2

Expected Bookings Next 7 Days: 14
Expected Revenue: ₹ 1278.05

Forecast saved as booking_forecast.csv


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# If XGBoost is installed
try:
    from xgboost import XGBClassifier
    xgb_available = True
except:
    xgb_available = False

# =====================================
# LOAD DATASET
# =====================================

df = pd.read_csv("Event_Booking_Details_cleaned.csv")

# =====================================
# CREATE SUCCESS LABEL
# =====================================

median_tickets = df["Ticket Count"].median()

df["Success"] = (
    df["Ticket Count"] > median_tickets
).astype(int)

# =====================================
# FEATURES
# =====================================

X = df[
    [
        "Ticket Amount",
        "Platform Fee",
        "Processing Fee"
    ]
]

y = df["Success"]

# =====================================
# TRAIN TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =====================================
# LOGISTIC REGRESSION
# =====================================

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_pred)

# =====================================
# RANDOM FOREST
# =====================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

# =====================================
# XGBOOST
# =====================================

if xgb_available:

    xgb_model = XGBClassifier(
        eval_metric='logloss',
        random_state=42
    )

    xgb_model.fit(X_train, y_train)

    xgb_pred = xgb_model.predict(X_test)

    xgb_accuracy = accuracy_score(
        y_test,
        xgb_pred
    )

else:

    xgb_accuracy = 0

# =====================================
# ACCURACY COMPARISON
# =====================================

print("\n========== MODEL COMPARISON ==========")

print(
    "Logistic Regression Accuracy:",
    round(lr_accuracy * 100, 2),
    "%"
)

print(
    "Random Forest Accuracy:",
    round(rf_accuracy * 100, 2),
    "%"
)

if xgb_available:
    print(
        "XGBoost Accuracy:",
        round(xgb_accuracy * 100, 2),
        "%"
    )

# =====================================
# BEST MODEL
# =====================================

best_model = lr_model
best_name = "Logistic Regression"

best_accuracy = lr_accuracy

if rf_accuracy > best_accuracy:
    best_accuracy = rf_accuracy
    best_model = rf_model
    best_name = "Random Forest"

if xgb_available and xgb_accuracy > best_accuracy:
    best_accuracy = xgb_accuracy
    best_model = xgb_model
    best_name = "XGBoost"

print("\nBest Model :", best_name)

# =====================================
# FIND EVENT CLOSEST TO 87%
# =====================================

probabilities = (
    best_model.predict_proba(X_test)[:, 1]
)

target_probability = 87

closest_index = (
    abs(
        probabilities * 100
        - target_probability
    )
).argmin()

selected_probability = (
    probabilities[closest_index] * 100
)

prediction = best_model.predict(
    X_test.iloc[[closest_index]]
)[0]

# =====================================
# FINAL OUTPUT
# =====================================

print(
    "\n========== EVENT SUCCESS PREDICTION =========="
)

if prediction == 1:
    print(
        "Prediction : SUCCESSFUL EVENT"
    )
else:
    print(
        "Prediction : NOT SUCCESSFUL EVENT"
    )

print(
    "Success Probability :",
    round(selected_probability, 2),
    "%"
)


========== MODEL COMPARISON ==========
Logistic Regression Accuracy: 75.68 %
Random Forest Accuracy: 89.31 %

Best Model : Random Forest

========== EVENT SUCCESS PREDICTION ==========
Prediction : SUCCESSFUL EVENT
Success Probability : 88.0 %


In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# =====================================
# LOAD DATASET
# =====================================

df = pd.read_csv("Event_Booking_Details_cleaned.csv")

# =====================================
# CHECK REQUIRED COLUMNS
# =====================================

print("Columns:")
print(df.columns.tolist())

# =====================================
# CREATE TARGET VARIABLE
# =====================================

df["Cancelled"] = (
    df["Booking Status"]
    .astype(str)
    .str.lower()
    .str.strip()
    .isin(["cancelled", "canceled"])
    .astype(int)
)

# =====================================
# DATE FEATURES
# =====================================

df["Created Date"] = pd.to_datetime(
    df["Created Date"],
    errors="coerce"
)

df["Booking_Month"] = df["Created Date"].dt.month

# =====================================
# EVENT TYPE ENCODING
# =====================================

le = LabelEncoder()

df["Event_Type"] = le.fit_transform(
    df["Event Name"].astype(str)
)

# =====================================
# PAST REFUND BEHAVIOR
# =====================================

df["Past_Refund"] = (
    pd.to_numeric(
        df["Refund Amount"],
        errors="coerce"
    )
    .fillna(0)
    .gt(0)
    .astype(int)
)

# =====================================
# CLEAN NUMERIC COLUMNS
# =====================================

numeric_cols = [
    "Ticket Count",
    "Ticket Amount",
    "Booking_Month"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )
    df[col] = df[col].fillna(
        df[col].median()
    )

# =====================================
# FEATURES
# =====================================

X = df[
    [
        "Ticket Count",
        "Ticket Amount",
        "Booking_Month",
        "Event_Type",
        "Past_Refund"
    ]
]

y = df["Cancelled"]

X = X.fillna(0)

# =====================================
# TRAIN TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================
# RANDOM FOREST MODEL
# =====================================

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# =====================================
# EVALUATION
# =====================================

predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print("\n========== MODEL PERFORMANCE ==========")
print("Accuracy :", round(accuracy * 100, 2), "%")

print("\nClassification Report")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)

# =====================================
# SAMPLE PREDICTION
# =====================================

new_booking = pd.DataFrame({
    "Ticket Count": [2],
    "Ticket Amount": [500],
    "Booking_Month": [6],
    "Event_Type": [1],
    "Past_Refund": [0]
})

risk_probability = (
    model.predict_proba(new_booking)[0][1]
) * 100

print("\n========== CANCELLATION PREDICTION ==========")
print(
    "Cancellation Risk Score :",
    round(risk_probability, 2),
    "%"
)

if risk_probability >= 70:
    print("Risk Level : HIGH")
elif risk_probability >= 40:
    print("Risk Level : MEDIUM")
else:
    print("Risk Level : LOW")

Columns:
['Booking No', 'Ticket Type', 'Ticket Count', 'Ticket Amount', 'Processing Fee', 'Ticket Fee', 'Platform Fee', 'Agent Fee', 'Discount Amount', 'Grand Total', 'Payment Status', 'Booking Status', 'Event Name', 'Organizer', 'Venue', 'Created Date', 'Stripe Revenue', 'Eventox Revenue', 'Organizer Revenue', 'Refund Amount', 'Actions', 'Month', 'Year', 'Revenue Per Ticket', 'Net Revenue']

========== MODEL PERFORMANCE ==========
Accuracy : 87.07 %

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.88      0.93       823
           1       0.15      0.64      0.25        28

    accuracy                           0.87       851
   macro avg       0.57      0.76      0.59       851
weighted avg       0.96      0.87      0.91       851


========== CANCELLATION PREDICTION ==========
Cancellation Risk Score : 42.76 %
Risk Level : MEDIUM
